# PS-S6E6 — Feature-rijke Baseline

Integreert alle 14 feature-groepen uit `xgb-v5-for-s6e6.ipynb` in een pure **pandas/numpy** pipeline (geen GPU vereist).

| Groep | Naam |
|---|---|
| 1 | Basis kleur-indices |
| 2 | Magnitude statistieken |
| 3 | Flux features |
| 4 | Hemelcoördinaten (trig + 3D) |
| 5 | Redshift features |
| 6 | Band × Redshift interacties |
| 7 | Absolute magnitude proxies |
| 8 | Kleurvlak (color plane) features |
| 9 | Spectrale categoricals + kruisfeatures |
| 10 | Floor-bin artifact features |
| 11 | Kleur-bin artifact features |
| 12 | Kwantielbin features |
| 13 | Frequentie features |
| 14 | Target Encoding (CV-veilig, binnen fold-loop) |


In [9]:
import gc
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)


In [10]:
SEED = 42
N_SPLITS = 5
TARGET = 'class'
BANDS = ['u', 'g', 'r', 'i', 'z']
CLASSES = ['GALAXY', 'QSO', 'STAR']
CLASS_TO_INT = {c: i for i, c in enumerate(CLASSES)}
INT_TO_CLASS = {i: c for c, i in CLASS_TO_INT.items()}
EPS = 1e-6
RAW_NUM_COLS = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']


## Data laden

In [11]:
import os

os.chdir("/mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-agils-ff-gpu/code/Users/agils/playground-series-s6e6/notebooks")

In [12]:
DATA_DIR = Path('../data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
train.columns = train.columns.str.lower()
test.columns  = test.columns.str.lower()

y        = train[TARGET].map(CLASS_TO_INT).astype('int8')
test_ids = test['id'].copy()
n_train  = len(train)

print(f'train: {train.shape}  test: {test.shape}')
print(train[TARGET].value_counts(normalize=True).round(4))


train: (577347, 12)  test: (247435, 11)
class
GALAXY    0.6538
QSO       0.2029
STAR      0.1433
Name: proportion, dtype: float64


## Feature Engineering

Alle functies accepteren een DataFrame en retourneren een kopie. Functies 1–13 worden vóór de CV-loop aangeroepen; functie 14 binnen de loop.

### Groep 1 — Basis kleur-indices

Alle paarsgewijze magnitude-verschillen van de 5 SDSS banden.

In [13]:
def add_color_indices(df):
    out = df.copy()
    color_pairs = [
        ('u','g'), ('g','r'), ('r','i'), ('i','z'),
        ('u','r'), ('u','i'), ('u','z'),
        ('g','i'), ('g','z'), ('r','z'),
    ]
    for a, b in color_pairs:
        out[f'{a}_{b}'] = (out[a] - out[b]).astype('float32')
    return out


### Groep 2 — Magnitude statistieken

Statistieken over de 5 banden: gemiddelde, std, min/max/range, helling (lineaire fit), en spectrale krommingen.

In [14]:
def add_magnitude_stats(df):
    out = df.copy()
    bv = out[BANDS].values.astype('float32')

    out['mag_mean']   = bv.mean(axis=1).astype('float32')
    out['mag_std']    = bv.std(axis=1, ddof=1).astype('float32')
    out['mag_min']    = bv.min(axis=1).astype('float32')
    out['mag_max']    = bv.max(axis=1).astype('float32')
    out['mag_range']  = (out['mag_max'] - out['mag_min']).astype('float32')
    out['mag_argmin'] = bv.argmin(axis=1).astype('int16')
    out['mag_argmax'] = bv.argmax(axis=1).astype('int16')

    # lineaire helling over bandindex
    x = np.arange(len(BANDS), dtype='float32')
    x_c = x - x.mean()
    out['mag_slope'] = (
        ((bv - bv.mean(axis=1, keepdims=True)) @ x_c) / (x_c ** 2).sum()
    ).astype('float32')

    # spectrale krommingen
    out['mag_curvature']  = (out['u'] - 2 * out['r'] + out['z']).astype('float32')
    out['blue_curvature'] = (out['u'] - 2 * out['g'] + out['r']).astype('float32')
    out['red_curvature']  = (out['r'] - 2 * out['i'] + out['z']).astype('float32')
    return out


### Groep 3 — Flux features

Magnitudes omrekenen naar flux: `flux = 10^(−0.4 × mag)`. Statistieken over de flux-vector.

In [15]:
def add_flux_features(df):
    out = df.copy()
    flux_arrays = []
    for b in BANDS:
        clipped = np.clip(out[b].values.astype('float32'), -30, 30)
        flux = (10.0 ** (-0.4 * clipped)).astype('float32')
        out[f'flux_{b}'] = flux
        flux_arrays.append(flux)
    fm = np.stack(flux_arrays, axis=1)
    out['flux_mean']  = fm.mean(axis=1).astype('float32')
    out['flux_std']   = fm.std(axis=1, ddof=1).astype('float32')
    out['flux_min']   = fm.min(axis=1).astype('float32')
    out['flux_max']   = fm.max(axis=1).astype('float32')
    out['flux_range'] = (out['flux_max'] - out['flux_min']).astype('float32')
    return out


### Groep 4 — Hemelcoördinaten (trig + 3D)

Ra/Dec (α/δ) omzetten naar sinus/cosinus en Cartesische 3D-coordinaten op de eenheidsbol.

In [16]:
def add_sky_coords(df):
    out = df.copy()
    alpha_rad = np.deg2rad(out['alpha'].values.astype('float32'))
    delta_rad = np.deg2rad(out['delta'].values.astype('float32'))
    out['alpha_sin'] = np.sin(alpha_rad).astype('float32')
    out['alpha_cos'] = np.cos(alpha_rad).astype('float32')
    out['delta_sin'] = np.sin(delta_rad).astype('float32')
    out['delta_cos'] = np.cos(delta_rad).astype('float32')
    cos_d = np.cos(delta_rad)
    out['sky_x'] = (cos_d * np.cos(alpha_rad)).astype('float32')
    out['sky_y'] = (cos_d * np.sin(alpha_rad)).astype('float32')
    out['sky_z'] = np.sin(delta_rad).astype('float32')
    return out


### Groep 5 — Redshift features

Absolute waarde, log-transformatie, teken-vlag, afstandsmodulus-proxy en fysische redshift-bin.

In [17]:
def add_redshift_features(df):
    out = df.copy()
    rs = out['redshift'].astype('float32')
    rs_abs = rs.abs()
    out['redshift_abs']          = rs_abs
    out['redshift_log1p_abs']    = np.log1p(rs_abs).astype('float32')
    out['redshift_is_neg']       = (rs < 0).astype('int8')
    out['redshift_distmod_proxy'] = (5.0 * np.log10(rs_abs + EPS)).astype('float32')
    # Fysische redshift-bins: <0.05, 0.05-0.10, 0.10-0.30, 0.30-0.60, >=0.60
    bins   = [-np.inf, 0.05, 0.10, 0.30, 0.60, np.inf]
    labels = ['z0', 'z1', 'z2', 'z3', 'z4']
    out['redshift_phys_bin'] = pd.cut(rs, bins=bins, labels=labels).astype('str')
    return out


### Groep 6 — Band × Redshift interacties

Producten band×redshift, ratio's band/|redshift|, kleur×redshift en kleur/|redshift|.

In [18]:
def add_redshift_interactions(df):
    out = df.copy()
    rs     = out['redshift'].astype('float32')
    rs_abs = rs.abs()
    for b in BANDS:
        out[f'redshift_{b}']     = (rs * out[b]).astype('float32')
        out[f'{b}_over_redshift'] = (out[b] / (rs_abs + EPS)).astype('float32')
    for c in ['u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'r_z', 'g_i']:
        if c in out.columns:
            out[f'{c}_x_redshift'] = (out[c] * rs).astype('float32')
    for c in ['u_g', 'g_r', 'r_i', 'i_z']:
        if c in out.columns:
            out[f'{c}_per_redshift'] = (out[c] / (rs_abs + EPS)).astype('float32')
    return out


### Groep 7 — Absolute magnitude proxies

Apparent magnitude gecorrigeerd voor afstand via `m − distmod_proxy`. Geeft een proxy voor de intrinsieke helderheid.

In [19]:
def add_absmag_proxies(df):
    out = df.copy()
    distmod = out['redshift_distmod_proxy'].astype('float32')
    for b in BANDS:
        out[f'{b}_absmag_proxy'] = (out[b] - distmod).astype('float32')
    if 'mag_mean' in out.columns:
        out['mag_mean_absmag_proxy'] = (out['mag_mean'] - distmod).astype('float32')
    for c in ['u_g', 'g_r', 'r_z', 'u_r', 'r_i']:
        if c in out.columns:
            out[f'{c}_abs'] = out[c].abs().astype('float32')
    return out


### Groep 8 — Kleurvlak (color plane) features

Radiaal en hoekcoördinaten in het (u−g, g−r) en (r−i, i−z) kleurvlak.

In [20]:
def add_color_plane_features(df):
    out = df.copy()
    if 'u_g' in out.columns and 'g_r' in out.columns:
        ug = out['u_g'].astype('float32')
        gr = out['g_r'].astype('float32')
        out['color_plane_radius_ug_gr'] = np.sqrt(ug**2 + gr**2).astype('float32')
        out['color_plane_angle_ug_gr']  = np.arctan2(ug, gr + EPS).astype('float32')
    if 'r_i' in out.columns and 'i_z' in out.columns:
        ri = out['r_i'].astype('float32')
        iz = out['i_z'].astype('float32')
        out['color_plane_radius_ri_iz'] = np.sqrt(ri**2 + iz**2).astype('float32')
        out['color_plane_angle_ri_iz']  = np.arctan2(ri, iz + EPS).astype('float32')
    return out


### Groep 9 — Spectrale categoricals + kruisfeatures

Berekent spectrale type en sterrenpopulatie direct uit fotometrie, voegt ordinale codering toe en maakt kruisproducten.

In [21]:
def _spectral_type_from_gr(r_minus_g):
    """Spectrale klasse afgeleid van r−g (zoals in de originele SDSS pipeline)."""
    return pd.cut(
        r_minus_g,
        bins=[-np.inf, -1.0, -0.5, 0.0, np.inf],
        labels=['M', 'G/K', 'A/F', 'O/B'],
    ).astype('str')


def _galaxy_population_from_ur(u_minus_r):
    """Sterrenstelsel-populatie via u−r kleurscheiding bij 2.22."""
    return pd.cut(
        u_minus_r,
        bins=[-np.inf, 2.2, np.inf],
        labels=['Blue_Cloud', 'Red_Sequence'],
    ).astype('str')


SPECTRAL_ORD_MAP = {'O/B': 0, 'A/F': 1, 'G/K': 2, 'M': 3,
                    'O': 0, 'B': 0, 'A': 1, 'F': 1, 'G': 2, 'K': 2}


def add_spectral_cross_features(df):
    out = df.copy()

    # berekende versies op basis van fotometrie
    if 'r_g' not in out.columns and 'g_r' in out.columns:
        r_minus_g = -out['g_r']
    elif 'r_g' in out.columns:
        r_minus_g = out['r_g']
    else:
        r_minus_g = out['r'] - out['g']
    out['spectral_type_calc']     = _spectral_type_from_gr(r_minus_g.astype('float32'))
    out['galaxy_population_calc'] = _galaxy_population_from_ur(
        out['u_r'].astype('float32') if 'u_r' in out.columns else (out['u'] - out['r']).astype('float32')
    )

    # ordinale codering
    out['spectral_ord'] = out['spectral_type'].map(SPECTRAL_ORD_MAP).fillna(-1).astype('float32')

    # g−r kleurbin (5 klassen)
    if 'g_r' in out.columns:
        out['g_r_color_bin'] = pd.cut(
            out['g_r'].astype('float32'),
            bins=[-np.inf, 0.0, 0.4, 0.8, 1.2, np.inf],
            labels=['gr0', 'gr1', 'gr2', 'gr3', 'gr4'],
        ).astype('str')

    # kruisfeatures als string-concatenaties
    def xcat(a, b):
        return out[a].astype('str') + '__' + out[b].astype('str')

    out['spectral_x_pop']         = xcat('spectral_type', 'galaxy_population')
    out['spectral_calc_x_pop_calc'] = xcat('spectral_type_calc', 'galaxy_population_calc')
    if 'redshift_phys_bin' in out.columns:
        out['redshift_phys_x_spectral']   = xcat('redshift_phys_bin', 'spectral_type')
        out['redshift_phys_x_pop']        = xcat('redshift_phys_bin', 'galaxy_population')
        if 'g_r_color_bin' in out.columns:
            out['redshift_phys_x_g_r_color'] = xcat('redshift_phys_bin', 'g_r_color_bin')
    return out


### Groep 10 — Floor-bin artifact features

Vloer van elke numerieke kolom als categorie-string. Vangt fijn-geraste patroonstructuren (telescoop-rasters, afrondingen) in de data.

In [22]:
def add_floor_artifacts(df):
    out = df.copy()
    for c in RAW_NUM_COLS:
        vals = pd.to_numeric(out[c], errors='coerce')
        na_mask = vals.isna()
        floored = np.floor(vals.fillna(0)).astype('int32').astype('str')
        floored[na_mask] = '__NA__'
        out[f'art_{c}_floor'] = floored
    out['art_alpha_floor_x_delta_floor'] = (
        out['art_alpha_floor'] + '__' + out['art_delta_floor']
    )
    out['art_u_floor_x_z_floor'] = (
        out['art_u_floor'] + '__' + out['art_z_floor']
    )
    return out


### Groep 11 — Kleur-bin artifact features

Vaste bin-breedtes op kleuren en posities. Levert categorische kenmerken die periodieke structuren opvangen.

In [23]:
COLOR_BIN_SPECS = [
    ('u_g',      2.0,  'half'),
    ('g_r',      2.0,  'half'),
    ('r_i',      2.0,  'half'),
    ('i_z',      2.0,  'half'),
    ('u_r',      1.0,  'one'),
    ('redshift', 10.0, 'tenth'),
    ('alpha',    0.2,  'deg5'),
    ('delta',    0.2,  'deg5'),
]


def add_color_bin_artifacts(df):
    out = df.copy()
    for col, scale, tag in COLOR_BIN_SPECS:
        if col not in out.columns:
            continue
        vals = pd.to_numeric(out[col], errors='coerce')
        na_mask = vals.isna()
        binned = np.floor(vals.fillna(0) * scale).astype('int32').astype('str')
        binned[na_mask] = '__NA__'
        name = f'art_{col}_{tag}'
        out[name] = binned
    # paar-kruisfeatures
    pairs = [
        ('art_u_g_half',    'art_redshift_tenth', 'art_u_g_half_x_redshift_tenth'),
        ('art_g_r_half',    'art_redshift_tenth', 'art_g_r_half_x_redshift_tenth'),
        ('art_alpha_deg5',  'art_delta_deg5',     'art_alpha_deg5_x_delta_deg5'),
    ]
    for a, b, name in pairs:
        if a in out.columns and b in out.columns:
            out[name] = out[a] + '__' + out[b]
    return out


### Groep 12 — Kwantielbin features

Bins bepaald op basis van kwantielen van train+test (geen target → geen leakage). Drie granulariteiten: q=16, 64, 256. Plus drie kruisfeatures.

In [24]:
QBIN_BASE_COLS = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
                  'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'mag_mean', 'mag_range']
QBIN_Qs = [16, 64, 256]


def add_quantile_bins(df, ref_mask):
    """
    ref_mask: boolean array, True for rows die de kwantiel-referentie bepalen
              (train+test, niet orig).
    """
    out = df.copy()
    new_cols = []
    for c in QBIN_BASE_COLS:
        if c not in out.columns:
            continue
        vals = pd.to_numeric(out[c], errors='coerce').values.astype('float64')
        ref  = vals[ref_mask]
        ref  = ref[~np.isnan(ref)]
        for q in QBIN_Qs:
            name = f'{c}_qbin{q}'
            probs = np.linspace(0, 1, q + 1)
            bins  = np.unique(np.quantile(ref, probs))
            if len(bins) <= 1:
                out[name] = '-1'
            else:
                nan_mask = np.isnan(vals)
                safe_vals = np.where(nan_mask, bins[0], vals)
                codes = np.searchsorted(bins, safe_vals, side='left') - 1
                codes = np.clip(codes, 0, len(bins) - 2)
                codes = np.where(nan_mask, -1, codes)
                out[name] = codes.astype('int16').astype('str')
            new_cols.append(name)
    # kruiskwantiel-bins
    cross_pairs = [
        ('alpha_qbin64',    'delta_qbin64',    'alpha_qbin64__x__delta_qbin64'),
        ('u_g_qbin64',      'g_r_qbin64',      'u_g_qbin64__x__g_r_qbin64'),
        ('redshift_qbin64', 'mag_mean_qbin64', 'redshift_qbin64__x__mag_mean_qbin64'),
    ]
    for a, b, name in cross_pairs:
        if a in out.columns and b in out.columns:
            out[name] = out[a].astype('str') + '__' + out[b].astype('str')
            new_cols.append(name)
    return out, new_cols


### Groep 13 — Frequentie features

Hoe vaak elke categorische waarde voorkomt in de referentieset (train+test). Geen target gebruikt → geen leakage.

In [25]:
def add_frequency_features(df, cols, ref_mask):
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            continue
        s = out[c].astype('str').fillna('__NA__')
        vc = s[ref_mask].value_counts(dropna=False)
        freq = s.map(vc).fillna(0).astype('float32')
        out[f'{c}_freq']      = freq
        out[f'{c}_freq_log1p'] = np.log1p(freq).astype('float32')
    return out


### Groep 14 — Target Encoding (CV-veilig)

Inner 7-fold TE met Laplace-smoothing (smooth=16). Wordt **binnen** de outer CV-fold aangeroepen zodat val/test nooit de training targets zien.

In [26]:
def add_fold_safe_target_encoding(X_train, y_train, X_val, X_test, te_cols,
                                   n_inner=7, smooth=16.0):
    X_train = X_train.copy()
    X_val   = X_val.copy()
    X_test  = X_test.copy()

    y_arr = np.asarray(y_train, dtype=np.int32)
    inner_skf = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=SEED + 177)
    inner_folds = np.empty(len(y_arr), dtype=np.int32)
    for fid, (_, va_idx) in enumerate(inner_skf.split(np.zeros(len(y_arr)), y_arr)):
        inner_folds[va_idx] = fid

    def _encode_from_stats(stats, keys, prior):
        merged = (
            pd.Series(keys, name='key')
            .to_frame()
            .join(stats, on='key', how='left')
        )
        n = merged['count'].fillna(0).values
        s = merged['sum'].fillna(0.0).values
        return ((s + smooth * prior) / (n + smooth)).astype(np.float32)

    for c in te_cols:
        if c not in X_train.columns:
            continue
        s_tr = X_train[c].astype(str).fillna('__NA__').values
        s_va = X_val[c].astype(str).fillna('__NA__').values
        s_te = X_test[c].astype(str).fillna('__NA__').values

        for cls_idx, cls_name in INT_TO_CLASS.items():
            y_bin = (y_arr == cls_idx).astype(np.float32)
            prior = float(y_bin.mean())

            # inner-fold OOF TE voor trainset
            tr_vals = np.full(len(X_train), prior, dtype=np.float32)
            for fid in range(n_inner):
                mask_tr = inner_folds != fid
                mask_va = inner_folds == fid
                tmp = pd.DataFrame({'key': s_tr[mask_tr], 'y': y_bin[mask_tr]})
                stats = tmp.groupby('key')['y'].agg(['sum', 'count'])
                tr_vals[mask_va] = _encode_from_stats(stats, s_tr[mask_va], prior)

            # volledige stats voor val + test
            tmp_full = pd.DataFrame({'key': s_tr, 'y': y_bin})
            stats_full = tmp_full.groupby('key')['y'].agg(['sum', 'count'])

            name = f'TE_{c}_{cls_name}'
            X_train[name] = tr_vals
            X_val[name]   = _encode_from_stats(stats_full, s_va, prior)
            X_test[name]  = _encode_from_stats(stats_full, s_te, prior)

    return X_train, X_val, X_test


## Feature matrix bouwen (pre-loop)

Groepen 1–13 worden op de gecombineerde train+test DataFrame toegepast.

In [27]:
drop_cols = [c for c in [TARGET, 'id'] if c in train.columns]
all_df = pd.concat(
    [train.drop(columns=drop_cols), test.drop(columns=['id'])],
    ignore_index=True,
)
ref_mask = np.ones(len(all_df), dtype=bool)   # train+test, geen orig

# Groepen 1–9 (numeriek + spectrale categoricals)
all_df = add_color_indices(all_df)
all_df = add_magnitude_stats(all_df)
all_df = add_flux_features(all_df)
all_df = add_sky_coords(all_df)
all_df = add_redshift_features(all_df)
all_df = add_redshift_interactions(all_df)
all_df = add_absmag_proxies(all_df)
all_df = add_color_plane_features(all_df)
all_df = add_spectral_cross_features(all_df)

# Groepen 10–11 (floor + color-bin artifacts)
all_df = add_floor_artifacts(all_df)
all_df = add_color_bin_artifacts(all_df)

# Groep 12 (kwantielbin)
all_df, qbin_cols = add_quantile_bins(all_df, ref_mask)

# Verzamel alle categorische kolommen voor groep 13 + TE
cat_cols = (
    ['spectral_type', 'galaxy_population',
     'spectral_type_calc', 'galaxy_population_calc',
     'spectral_x_pop', 'spectral_calc_x_pop_calc',
     'redshift_phys_bin', 'g_r_color_bin',
     'redshift_phys_x_spectral', 'redshift_phys_x_pop', 'redshift_phys_x_g_r_color'] +
    [f'art_{c}_floor' for c in RAW_NUM_COLS] +
    ['art_alpha_floor_x_delta_floor', 'art_u_floor_x_z_floor'] +
    [f'art_{col}_{tag}' for col, _, tag in COLOR_BIN_SPECS] +
    ['art_u_g_half_x_redshift_tenth', 'art_g_r_half_x_redshift_tenth',
     'art_alpha_deg5_x_delta_deg5'] +
    qbin_cols
)
cat_cols = [c for c in dict.fromkeys(cat_cols) if c in all_df.columns]

# Groep 13 (frequentie)
all_df = add_frequency_features(all_df, cat_cols, ref_mask=pd.Series(ref_mask))

# Splits terug
X         = all_df.iloc[:n_train].reset_index(drop=True)
X_test_df = all_df.iloc[n_train:].reset_index(drop=True)
del all_df; gc.collect()

# TE-kolomselectie: alleen cols die nuttig zijn (cardinality-filter)
MAX_CARD = 5000
te_cols = [
    c for c in cat_cols
    if c in X.columns and X[c].astype(str).nunique() <= MAX_CARD
]

print(f'Feature matrix: {X.shape}')
print(f'Categorische kolommen: {len(cat_cols)}')
print(f'TE-bron kolommen: {len(te_cols)}')


Feature matrix: (577347, 327)
Categorische kolommen: 80
TE-bron kolommen: 79


## Model parameters

Zet `MODEL_IN_USE` op `'XGB'` of `'CAT'`.

In [28]:
MODEL_IN_USE = 'XGB'

XGB_PARAMS = {
    'objective':          'multi:softprob',
    'num_class':          3,
    'tree_method':        'hist',
    'device':             'cuda',      # verander naar 'cuda' als GPU beschikbaar
    'n_estimators':       10000,
    'early_stopping_rounds': 100,
    'learning_rate':      0.05,
    'max_depth':          0,          # onbeperkt met lossguide
    'max_leaves':         64,
    'grow_policy':        'lossguide',
    'max_bin':            256,
    'min_child_weight':   10,
    'gamma':              0.2,
    'reg_alpha':          0.30,
    'reg_lambda':         4.0,
    'subsample':          0.82,
    'colsample_bytree':   0.74,
    'colsample_bylevel':  0.86,
    'enable_categorical': True,
    'random_state':       SEED,
    'n_jobs':             4,
}


## CV Loop met fold-veilige Target Encoding

In [29]:
y_arr       = y.to_numpy()
oof         = np.zeros((len(X), 3), dtype='float32')
test_preds  = np.zeros((len(X_test_df), 3), dtype='float32')
fold_scores = []

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_arr)), y_arr), start=1):
    print(f'\n===== Fold {fold}/{N_SPLITS} =====')
    X_tr = X.iloc[tr_idx].reset_index(drop=True)
    y_tr = pd.Series(y_arr[tr_idx])
    X_va = X.iloc[va_idx].reset_index(drop=True)
    y_va = pd.Series(y_arr[va_idx])
    X_te = X_test_df.copy()

    # Groep 14: fold-veilige TE
    X_tr, X_va, X_te = add_fold_safe_target_encoding(
        X_tr, y_tr, X_va, X_te, te_cols
    )

    # Categorische kolommen als category dtype (voor XGB enable_categorical).
    # Val/test krijgen dezelfde categorieset als train zodat onbekende waarden
    # NaN worden in plaats van een fout te geven.
    all_cat = [c for c in cat_cols if c in X_tr.columns]
    for c in all_cat:
        X_tr[c] = X_tr[c].astype('category')
        train_cats = X_tr[c].cat.categories
        X_va[c] = pd.Categorical(X_va[c], categories=train_cats)
        X_te[c] = pd.Categorical(X_te[c], categories=train_cats)

    train_w = compute_sample_weight(class_weight='balanced', y=y_tr)
    val_w   = compute_sample_weight(class_weight='balanced', y=y_va)

    if MODEL_IN_USE == 'XGB':
        model = XGBClassifier(**{**XGB_PARAMS, 'random_state': SEED + fold * 100})

        print('fitting model')
        model.fit(
            X_tr, y_tr,
            sample_weight=train_w,
            eval_set=[(X_va, y_va)],
            sample_weight_eval_set=[val_w],
            verbose=500,
        )

    oof[va_idx]  = model.predict_proba(X_va)
    test_preds  += model.predict_proba(X_te) / N_SPLITS

    score = balanced_accuracy_score(y_arr[va_idx], np.argmax(oof[va_idx], axis=1))
    fold_scores.append(score)
    best = getattr(model, 'best_iteration', '?')
    print(f'Fold {fold} balanced_accuracy={score:.6f}  best_iter={best}')

    del X_tr, X_va, X_te, model; gc.collect()

oof_score = balanced_accuracy_score(y_arr, np.argmax(oof, axis=1))
print(f'\nMean fold:  {np.mean(fold_scores):.6f}')
print(f'OOF totaal: {oof_score:.6f}')



===== Fold 1/5 =====
fitting model
[0]	validation_0-mlogloss:1.03334
[268]	validation_0-mlogloss:0.09983
Fold 1 balanced_accuracy=0.966136  best_iter=168

===== Fold 2/5 =====
fitting model
[0]	validation_0-mlogloss:1.03312
[280]	validation_0-mlogloss:0.10177
Fold 2 balanced_accuracy=0.965768  best_iter=180

===== Fold 3/5 =====
fitting model
[0]	validation_0-mlogloss:1.03349
[267]	validation_0-mlogloss:0.10370
Fold 3 balanced_accuracy=0.964961  best_iter=167

===== Fold 4/5 =====
fitting model
[0]	validation_0-mlogloss:1.03333
[267]	validation_0-mlogloss:0.10278
Fold 4 balanced_accuracy=0.964336  best_iter=167

===== Fold 5/5 =====
fitting model
[0]	validation_0-mlogloss:1.03339
[269]	validation_0-mlogloss:0.10240
Fold 5 balanced_accuracy=0.965325  best_iter=169

Mean fold:  0.965305
OOF totaal: 0.965305


## Artifacts opslaan

In [30]:
SUBMISSIONS_DIR = Path('../submissions')
SUBMISSIONS_DIR.mkdir(exist_ok=True)

# OOF
oof_df = pd.DataFrame(oof, columns=['p_galaxy', 'p_qso', 'p_star'])
oof_df['class'] = y_arr
oof_df.to_parquet(SUBMISSIONS_DIR / 'oof_baseline_features.parquet', index=False)

# Test
test_df = pd.DataFrame(test_preds, columns=['p_galaxy', 'p_qso', 'p_star'])
test_df.to_parquet(SUBMISSIONS_DIR / 'oof_test_baseline_features.parquet', index=False)

# Submission
pred_labels = [INT_TO_CLASS[i] for i in np.argmax(test_preds, axis=1)]
sub = sample_submission.copy()
sub[TARGET] = pred_labels
sub.to_csv(SUBMISSIONS_DIR / f'submission_baseline_features_{oof_score:.10f}.csv', index=False)

print(f'Opgeslagen: oof {oof_df.shape}, test {test_df.shape}, sub {sub.shape}')
sub.head()


Opgeslagen: oof (577347, 4), test (247435, 3), sub (247435, 2)


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
